<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG

An advanced Retrieval-Augmented Generation (RAG) system that uses intelligent agents to retrieve and synthesize information from documents.

## Dependencies Installation

Install required packages for the RAG system.

**Documentation:**
- [LangGraph](https://langchain-ai.github.io/langgraph/) - Framework for building custom agents with low-level control
- [LangChain](https://python.langchain.com/docs/get_started/introduction) - Framework to quick start agents with any model provider
- [Qdrant](https://qdrant.tech/documentation/) - Vector database for similarity search
- [Gradio](https://www.gradio.app/docs) - Web interface for ML models

In [ ]:
import os
import subprocess

IN_COLAB_SETUP = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB_SETUP and not os.path.exists("requirements.txt"):
    repo_dir = "agentic-rag-for-dummies"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "https://github.com/GiovanniPasq/agentic-rag-for-dummies.git"], check=True)
    os.chdir(repo_dir)

REQ_PATH = "requirements.txt" if os.path.exists("requirements.txt") else "../requirements.txt"
!pip install -qU -r {REQ_PATH}

## 1. Environment Configuration

Set up directory structure and environment variables for document processing.

**What it does:**
- Creates directories for storing PDFs, Markdown files, and parent chunks
- Defines collection names for the vector database

In [ ]:
import os

# Configuration
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ

DOCS_DIR = "docs" if IN_COLAB else "../docs"                    # Directory containing your pdf files
MARKDOWN_DIR = "markdown_docs" if IN_COLAB else "../markdown_docs"  # Directory containing the pdfs converted to markdown
PARENT_STORE_PATH = "parent_store" if IN_COLAB else "../parent_store"  # Directory for parent chunk JSON files
QDRANT_DB_PATH = "qdrant_db" if IN_COLAB else "../qdrant_db"
CHILD_COLLECTION = "document_child_chunks"
RETRIEVAL_SCORE_THRESHOLD = 0.4
DEFAULT_RETRIEVAL_K = 7
CHILD_CHUNK_SEPARATOR = "\n\n<CHILD_CHUNK_BOUNDARY>\n\n"

os.makedirs(DOCS_DIR, exist_ok=True)
os.makedirs(MARKDOWN_DIR, exist_ok=True)
os.makedirs(PARENT_STORE_PATH, exist_ok=True)

## 2. LLM Initialization

Initialize the Large Language Model that will power the conversational agent.

**What it does:**
- Configures the LLM using Ollama (local inference)
- Alternative example provided for Google Gemini

> **Hosted Colab note:** Standard Colab does not include a running Ollama server. Use the cloud-provider example with its integration package and API key, or run the default Ollama cell locally.

**Documentation:**
- [LangChain Ollama](https://python.langchain.com/docs/integrations/chat/ollama)
- [LangChain Google GenAI](https://python.langchain.com/docs/integrations/chat/google_generative_ai)

In [ ]:
from langchain_ollama import ChatOllama

# Initialize LLM
LLM_SEED = 42
llm = ChatOllama(model="granite4.1:8b", temperature=0, seed=LLM_SEED)

# Alternative (example): Google Gemini
# from langchain_google_genai import ChatGoogleGenerativeAI
# os.environ["GOOGLE_API_KEY"] = "your-api-key-here"
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

## 3. Embeddings Setup

Configure embedding models for semantic search using hybrid retrieval (dense + sparse).

**What it does:**
- **Dense embeddings**: Capture semantic meaning using neural networks
- **Sparse embeddings**: Provide keyword-based matching (BM25 algorithm)

**Documentation:**
- [HuggingFace Embeddings](https://python.langchain.com/docs/integrations/text_embedding/huggingfacehub)
- [FastEmbed Sparse](https://qdrant.github.io/fastembed/)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant.fastembed_sparse import FastEmbedSparse

# Dense embeddings for semantic understanding.
dense_embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

# Sparse embeddings for keyword matching
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

## 4. Vector Database Configuration

Set up Qdrant vector database for storing and retrieving document embeddings.

**What it does:**
- Initializes local Qdrant client with file-based storage
- Creates collections with both dense and sparse vector configurations
- Enables hybrid search capabilities

**Documentation:**
- [LangChain Qdrant](https://python.langchain.com/docs/integrations/vectorstores/qdrant)

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant.qdrant import RetrievalMode

# Initialize Qdrant client (local file-based storage)
client = QdrantClient(path=QDRANT_DB_PATH)

# Get embedding dimension
embedding_dimension = len(dense_embeddings.embed_query("test"))

def get_collection_vector_size(collection_info):
    vectors_config = collection_info.config.params.vectors
    if hasattr(vectors_config, "size"):
        return vectors_config.size
    if isinstance(vectors_config, dict) and vectors_config:
        first_vector = next(iter(vectors_config.values()))
        return getattr(first_vector, "size", None)
    return None

def ensure_collection(collection_name):
    """Create Qdrant collection if needed and validate embedding dimensions."""
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=qmodels.VectorParams(
                size=embedding_dimension,
                distance=qmodels.Distance.COSINE
            ),
            sparse_vectors_config={
                "sparse": qmodels.SparseVectorParams()
            },
        )
        print(f"✓ Created collection: {collection_name}")
        return

    collection_info = client.get_collection(collection_name)
    existing_dimension = get_collection_vector_size(collection_info)
    if existing_dimension and existing_dimension != embedding_dimension:
        raise ValueError(
            f"Collection '{collection_name}' uses dense vector size {existing_dimension}, "
            f"but the current embedding model produces {embedding_dimension}. "
            "Clear qdrant_db and re-index documents after changing embeddings."
        )

    print(f"✓ Collection already exists: {collection_name}")

## 5. PDF to Markdown Conversion

Convert PDF documents to Markdown format for better text extraction and processing.

**What it does:**
- Uses PyMuPDF to extract text from PDFs
- Converts documents to clean Markdown format
- Handles encoding issues and removes images
- Skips already converted files unless overwrite is enabled

**Note:** For more details on PDF conversion, refer to the `pdf_to_markdown.ipynb` notebook in the repository.

**Optional:** 🐿️ [**Chunky**](https://github.com/GiovanniPasq/chunky) is an open-source toolkit for reliable RAG pipelines: convert PDFs to Markdown, clean documents, inspect chunks, compare chunking strategies, and enrich metadata before building the vector store.

**Documentation:**
- [PyMuPDF](https://pymupdf.readthedocs.io/)
- [PyMuPDF4LLM](https://github.com/pymupdf/PyMuPDF4LLM)

In [ ]:
import os
import pymupdf.layout
import pymupdf4llm
from pathlib import Path
import glob

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def pdf_to_markdown(pdf_path, output_dir):
    doc = pymupdf.open(pdf_path)
    md = pymupdf4llm.to_markdown(doc, header=False, footer=False, page_separators=True, ignore_images=True, write_images=False, image_path=None)
    md_cleaned = md.encode('utf-8', errors='surrogatepass').decode('utf-8', errors='ignore')
    output_path = Path(output_dir) / Path(doc.name).stem
    Path(output_path).with_suffix(".md").write_bytes(md_cleaned.encode('utf-8'))

def pdfs_to_markdowns(path_pattern, overwrite: bool = False):
    output_dir = Path(MARKDOWN_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for pdf_path in map(Path, glob.glob(path_pattern)):
        md_path = (output_dir / pdf_path.stem).with_suffix(".md")
        if overwrite or not md_path.exists():
            pdf_to_markdown(pdf_path, output_dir)

pdfs_to_markdowns(f"{DOCS_DIR}/*.pdf")

## 6. Document Indexing

Implement parent-child chunking strategy for optimal retrieval performance.

**What it does:**
- Splits documents into hierarchical chunks (parent and child)
- **Parent chunks**: Large context windows (2000-4000 chars) stored as JSON
- **Child chunks**: Small searchable units (500 chars) stored in Qdrant
- Merges small chunks and splits large ones for consistency
- Copies each parent ID into child metadata so the agent can load the larger context

**Chunking Strategy:**
1. Split by Markdown headers (#, ##, ###)
2. Merge chunks smaller than 2000 characters
3. Split chunks larger than 4000 characters
4. Create child chunks (500 chars) from each parent
5. Store parent chunks in JSON files
6. Index child chunks in vector database

**Documentation:**
- [LangChain Text Splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- [LangChain Split Markdown](https://docs.langchain.com/oss/python/integrations/splitters/markdown_header_metadata_splitter)


In [ ]:
import os
import glob
import json
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

if client.collection_exists(CHILD_COLLECTION):
    print(f"Removing existing Qdrant collection: {CHILD_COLLECTION}")
    client.delete_collection(CHILD_COLLECTION)
    ensure_collection(CHILD_COLLECTION)
else:
    ensure_collection(CHILD_COLLECTION)

child_vector_store = QdrantVectorStore(
    client=client,
    collection_name=CHILD_COLLECTION,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode=RetrievalMode.HYBRID,
    sparse_vector_name="sparse"
)

def index_documents():
    headers_to_split_on = [("#", "H1"), ("##", "H2"), ("###", "H3")]
    parent_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on, strip_headers=False)
    child_chunk_size = 500
    child_chunk_overlap = 100
    min_parent_size = 2000
    max_parent_size = 4000
    if min_parent_size <= 0 or max_parent_size < min_parent_size:
        raise ValueError("Parent chunk sizes must be positive and min_parent_size <= max_parent_size.")
    if not 0 <= child_chunk_overlap < child_chunk_size:
        raise ValueError("child_chunk_overlap must be smaller than child_chunk_size.")
    child_splitter = RecursiveCharacterTextSplitter(
        chunk_size=child_chunk_size,
        chunk_overlap=child_chunk_overlap,
    )

    all_parent_pairs, all_child_chunks = [], []
    md_files = sorted(glob.glob(os.path.join(MARKDOWN_DIR, "*.md")))

    if not md_files:
        print(f"⚠️  No .md files found in {MARKDOWN_DIR}/")
        return

    for doc_path_str in md_files:
        doc_path = Path(doc_path_str)
        print(f"📄 Processing: {doc_path.name}")

        try:
            with open(doc_path, "r", encoding="utf-8") as f:
                md_text = f.read()
        except Exception as e:
            print(f"❌ Error reading {doc_path.name}: {e}")
            continue

        parent_chunks = parent_splitter.split_text(md_text)
        merged_parents = merge_small_parents(parent_chunks, min_parent_size)
        split_parents = split_large_parents(merged_parents, max_parent_size, child_chunk_overlap)
        cleaned_parents = clean_small_chunks(split_parents, min_parent_size, max_parent_size)
        if any(len(chunk.page_content) > max_parent_size for chunk in cleaned_parents):
            raise ValueError("Parent chunking produced an oversized chunk.")

        for i, p_chunk in enumerate(cleaned_parents):
            parent_id = f"{doc_path.stem}_p{i}"
            p_chunk.metadata.update({"source": doc_path.stem + ".pdf", "parent_id": parent_id})
            all_parent_pairs.append((parent_id, p_chunk))
            children = child_splitter.split_documents([p_chunk])
            all_child_chunks.extend(children)

    if not all_child_chunks:
        print("⚠️ No child chunks to index")
        return

    print(f"\n🔍 Indexing {len(all_child_chunks)} child chunks into Qdrant...")
    try:
        child_vector_store.add_documents(all_child_chunks)
        print("✓ Child chunks indexed successfully")
    except Exception as e:
        print(f"❌ Error indexing child chunks: {e}")
        return

    print(f"💾 Saving {len(all_parent_pairs)} parent chunks to JSON...")
    for item in os.listdir(PARENT_STORE_PATH):
        os.remove(os.path.join(PARENT_STORE_PATH, item))

    for parent_id, doc in all_parent_pairs:
        doc_dict = {"page_content": doc.page_content, "metadata": doc.metadata}
        filepath = os.path.join(PARENT_STORE_PATH, f"{parent_id}.json")
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

def merge_metadata(target, source, prepend=False):
    for key, value in source.items():
        if key not in target:
            target[key] = value
        else:
            first, second = (value, target[key]) if prepend else (target[key], value)
            values = [
                item.strip()
                for raw in (first, second)
                for item in str(raw).split(" -> ")
                if item.strip()
            ]
            target[key] = " -> ".join(dict.fromkeys(values))

def merge_small_parents(chunks, min_size):
    if not chunks:
        return []

    merged, current = [], None

    for chunk in chunks:
        if current is None:
            current = chunk
        else:
            current.page_content += "\n\n" + chunk.page_content
            merge_metadata(current.metadata, chunk.metadata)

        if len(current.page_content) >= min_size:
            merged.append(current)
            current = None

    if current:
        if merged:
            merged[-1].page_content += "\n\n" + current.page_content
            merge_metadata(merged[-1].metadata, current.metadata)
        else:
            merged.append(current)

    return merged

def split_large_parents(chunks, max_size, overlap):
    split_chunks = []

    for chunk in chunks:
        if len(chunk.page_content) <= max_size:
            split_chunks.append(chunk)
        else:
            large_splitter = RecursiveCharacterTextSplitter(
                chunk_size=max_size,
                chunk_overlap=overlap
            )
            sub_chunks = large_splitter.split_documents([chunk])
            split_chunks.extend(sub_chunks)

    return split_chunks

def rebalance_pair(first, second, min_size, max_size):
    combined = first.page_content.rstrip() + "\n\n" + second.page_content.lstrip()
    lower = max(1, len(combined) - max_size)
    upper = min(max_size, len(combined) - 1)
    if len(combined) >= 2 * min_size:
        lower = max(lower, min_size)
        upper = min(upper, len(combined) - min_size)
    preferred = min(max(len(combined) // 2, lower), upper)

    split_at = preferred
    for separator in ("\n\n", "\n", " "):
        before = combined.rfind(separator, lower, preferred + 1)
        after = combined.find(separator, preferred, upper + 1)
        if before >= lower:
            split_at = before
            break
        if after != -1:
            split_at = after
            break

    left_text = combined[:split_at].rstrip()
    right_text = combined[split_at:].lstrip()
    if len(combined) >= 2 * min_size and (len(left_text) < min_size or len(right_text) < min_size):
        split_at = preferred
        left_text, right_text = combined[:split_at], combined[split_at:]
    if not left_text or not right_text:
        return first, second

    metadata = dict(first.metadata)
    merge_metadata(metadata, second.metadata)
    first.page_content, first.metadata = left_text, dict(metadata)
    second.page_content, second.metadata = right_text, dict(metadata)
    return first, second

def clean_small_chunks(chunks, min_size, max_size):
    cleaned = []

    for i, chunk in enumerate(chunks):
        if len(chunk.page_content) < min_size:
            if cleaned and len(cleaned[-1].page_content) + 2 + len(chunk.page_content) <= max_size:
                cleaned[-1].page_content += "\n\n" + chunk.page_content
                merge_metadata(cleaned[-1].metadata, chunk.metadata)
            elif i < len(chunks) - 1 and len(chunk.page_content) + 2 + len(chunks[i + 1].page_content) <= max_size:
                chunks[i + 1].page_content = chunk.page_content + "\n\n" + chunks[i + 1].page_content
                merge_metadata(chunks[i + 1].metadata, chunk.metadata, prepend=True)
            else:
                cleaned.append(chunk)
        else:
            cleaned.append(chunk)

    for i, chunk in enumerate(cleaned):
        if len(chunk.page_content) >= min_size or len(cleaned) == 1:
            continue
        if i < len(cleaned) - 1:
            cleaned[i], cleaned[i + 1] = rebalance_pair(chunk, cleaned[i + 1], min_size, max_size)
        else:
            cleaned[i - 1], cleaned[i] = rebalance_pair(cleaned[i - 1], chunk, min_size, max_size)

    return cleaned

index_documents()

## 7. Tools Definition

Define retrieval tools that agents can use to search and retrieve document chunks.

**What it does:**
- **search_child_chunks**: Searches vector database for relevant small chunks
- **retrieve_parent_chunks**: Retrieves full context from parent chunk JSON files
- Binds tools to LLM for agentic function calling

**Two-stage retrieval:**
1. Agent searches child chunks (fast, semantic search)
2. Agent retrieves parent chunks for full context (when needed)

**Documentation:**
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)

In [ ]:
import json
from langchain_core.tools import tool

@tool
def search_child_chunks(query: str, limit: int = DEFAULT_RETRIEVAL_K) -> str:
    """Search document excerpts for evidence related to the user question.

    Use this as the first retrieval step. Results include parent IDs, file
    names, and short child-chunk excerpts. If excerpts are relevant but too
    fragmented to answer confidently, call retrieve_parent_chunks with the
    returned parent_id.

    Args:
        query: Focused search query with concrete keywords from the question.
        limit: Maximum number of child chunks to return.
    """
    try:
        results = child_vector_store.similarity_search(
            query,
            k=limit,
            score_threshold=RETRIEVAL_SCORE_THRESHOLD,
        )
        if not results:
            return "NO_RELEVANT_CHUNKS"

        return CHILD_CHUNK_SEPARATOR.join([
            f"Parent ID: {doc.metadata.get('parent_id', '')}\n"
            f"File Name: {doc.metadata.get('source', '')}\n"
            f"Content: {doc.page_content.strip()}"
            for doc in results
        ])

    except Exception as e:
        return f"RETRIEVAL_ERROR: {str(e)}"

@tool
def retrieve_parent_chunks(parent_id: str) -> str:
    """Retrieve the full parent chunk for a relevant child search result.

    Use this only after search_child_chunks returns a relevant parent_id and
    the child excerpt needs more surrounding context. Do not call this for
    parent IDs already available in compressed context.

    Args:
        parent_id: Parent chunk ID returned by search_child_chunks.
    """
    file_name = parent_id if parent_id.lower().endswith(".json") else f"{parent_id}.json"
    path = os.path.join(PARENT_STORE_PATH, file_name)

    if not os.path.exists(path):
        return "NO_PARENT_DOCUMENT"

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    return (
        f"Parent ID: {parent_id}\n"
        f"File Name: {data.get('metadata', {}).get('source', 'unknown')}\n"
        f"Content: {data.get('page_content', '').strip()}"
    )

# Bind tools to LLM
llm_with_tools = llm.bind_tools([search_child_chunks, retrieve_parent_chunks])

## 8. System Prompts

Define system prompts that guide agent behavior throughout the RAG pipeline.

**What it does:**
- **Conversation Summary**: Maintains a rolling summary while keeping only the last two user/assistant exchanges as raw chat history
- **Query Rewrite**: Rewrites unclear queries, splits multi-part questions, and incorporates follow-up context only when needed
- **Orchestrator**: Forces retrieval before answering, uses compressed memory to avoid redundant searches and parent chunk re-fetches
- **Fallback Response**: Synthesizes a best-effort answer from already-retrieved context when the operation limit is reached
- **Context Compression**: Compresses prior research into a structured, query-focused summary organized by source file, including a gaps section for missing information
- **Aggregation**: Merges multiple sub-answers into a single cohesive final response

**Key behaviors:**
- Query rewriting uses conversation context minimally; domain-specific terms are preserved as-is
- Agent checks compressed memory before searching to avoid re-querying covered aspects or re-fetching known parent IDs
- Retry with broader/rephrased query if no relevant documents are found
- Fallback Response activates at the operation limit, using only previously retrieved content
- Source attribution at the end of every response

In [ ]:
def get_conversation_summary_prompt() -> str:
    return """## Role
You are a compact memory manager for a retrieval-augmented chat assistant.

## Context
The input contains an existing rolling summary plus older user/assistant messages that will be removed from raw chat history.

## Instructions
- Merge the existing summary with the new older messages.
- Preserve context needed for future follow-up questions: topics, user preferences, important facts, unresolved questions, and referenced source file names.
- Discard greetings, tool calls, tool outputs, formatting chatter, duplicate details, and resolved misunderstandings.
- Keep the summary compact: 30-70 words unless more detail is essential.

## Output
Return exactly one merged summary and nothing else.
Do not include labels such as "Updated summary:", "Previous summary:", or "New messages:".
Do not include both old and new summaries.
If there is no meaningful context, return an empty string.
"""

def get_rewrite_query_prompt() -> str:
    return """## Role
You are a query rewriting specialist for document retrieval in a RAG system.

## Instructions
- Rewrite the current query so it is clear, self-contained, and useful for retrieval.
- Use the conversation summary and recent conversation only to resolve vague follow-ups that refer to prior context.
- When an unresolved query and one or more user clarifications are provided, combine all of them into one self-contained retrieval query.
- If the query is a follow-up, integrate only the minimal context needed to make it self-contained.
- Preserve product names, file names, versions, acronyms, numbers, and technical terms exactly.
- If the user asks about a named topic, product, file, acronym, term, or concept, treat the question as clear even if it is new.
- Standalone named terms, acronyms, or concepts are valid retrieval queries; do not require prior conversation context.
- Split only truly separate information needs, with a maximum of 3 rewritten questions.

## Clarification Boundary
Mark the query unclear only when it depends on an unresolved reference such as "it", "that", "this file", or "the previous one".
Do not mark a query unclear because the topic was not mentioned earlier.
Do not ask the user whether a new acronym or term is a typo; preserve it and search for it.

## Constraints
Do not add facts, expand acronyms, invent context, or broaden the user's meaning.
"""

def get_orchestrator_prompt() -> str:
    return """## Role
You are a document-grounded research assistant for an agentic RAG system. Your job is to answer using retrieved document evidence, not general knowledge.

## Available Context
- Current user question
- Optional compressed context from prior retrieval steps
- Tools for searching child chunks and loading full parent chunks

## Tool Guidance
- Search documents before answering unless compressed context already contains enough evidence.
- Use 'search_child_chunks' for missing or uncovered parts of the question.
- If searched or retrieved context is not useful, use the tools again with a different, simpler query or a more relevant parent chunk.
- Continue tool use until the available evidence is enough, tools stop adding useful information, or the operation limit is reached.
- Do not repeat search queries or parent IDs listed in compressed context.
- Do not retrieve the same parent ID twice.

## Response Framework
1. Check compressed context for already-known evidence and already-used searches or parents.
2. Search for missing evidence.
3. Retrieve parent chunks only when child excerpts are relevant but too fragmented.
4. Answer using the exact terms and scope in the retrieved evidence.
5. If evidence is incomplete, state the specific gap.

## Output
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- Do not mention internal tool calls or reasoning.
- When sources exist, end with a Sources section in exactly this format:
  Sources:
  - filename.ext
- Put each source filename on its own bullet line. Never write sources inline, such as "Sources: filename.pdf".
- Do not invent or infer source filenames.
- Strip descriptions after file names, including text in parentheses.
"""

def get_fallback_response_prompt() -> str:
    return """## Role
You are a constrained evidence synthesizer for a retrieval-augmented assistant after the research loop reached its limit.

## Available Context
- Compressed Research Context from earlier retrieval steps
- Retrieved Data from current tool outputs

## Instructions
- Use only explicit facts from the provided context.
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Prefer current Retrieved Data over compressed context if they conflict.
- If the answer is incomplete, mention only the missing parts that matter to the user query.
- Do not describe the retrieval process, limits, or internal reasoning.
- Be concise: answer in 1-3 short paragraphs or up to 5 bullets unless the user asks for detail.
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- End with a Sources section only when actual source file names are explicitly present in the context.
- Use exactly this format:
  Sources:
  - filename.ext
- Put each source filename on its own bullet line. Never write sources inline, such as "Sources: filename.pdf".
- Include only bare file names with extensions such as .pdf, .docx, .txt, or .md.
- Do not invent or infer source filenames.
"""

def get_context_compression_prompt() -> str:
    return """## Role
You are a research context compressor for an agentic RAG system.

## Instructions
- Keep only facts relevant to answering the user question.
- Preserve exact names, figures, versions, technical terms, configuration details, and source file names.
- Remove duplicates, tool chatter, search query wording, parent IDs, chunk IDs, and other internal identifiers.
- Organize findings by source file. Each source section heading must be the real filename found in retrieved data.
- Add a Gaps section only for missing information relevant to the question.
- Target 400-600 words. If there is too much content, keep the most answer-critical facts.

## Output
Return only Markdown in this structure:
# Research Context Summary

## Focus
[Brief technical restatement of the question]

## Structured Findings
For each source file, add a level-3 heading with its real filename and bullet the directly relevant facts below it.

## Gaps
- Missing or incomplete aspects
"""

def get_aggregation_prompt() -> str:
    return """## Role
You are a final-answer synthesizer for a retrieval-augmented assistant.

## Instructions
- Use only information present in the retrieved answers.
- Start directly with the substantive answer. Do not start with generic headings such as "Answer", "Final answer", or "Response".
- Preserve important names, numbers, versions, examples, and definitions.
- Do not expand acronyms or interpret terms unless the sources do it.
- If answers conflict, mention the conflict plainly.
- Be concise: answer in 1-3 short paragraphs or up to 5 bullets unless the user asks for detail.
- Provide the direct answer plus the key supporting details from retrieved evidence; avoid one-sentence fragments unless only one fact is available.
- End with a Sources section only when actual source file names are explicitly present in the retrieved answers.
- Use exactly this format:
  Sources:
  - filename.ext
- Put each source filename on its own bullet line. Never write sources inline, such as "Sources: filename.pdf".
- Include only bare file names with extensions such as .pdf, .docx, .txt, or .md.
- Do not invent or infer source filenames.
- If no useful information is available, say: "I couldn't find any information to answer your question in the available sources."
"""


## 9. State Definitions

Define state schemas for managing conversation flow and agent execution.

**What it does:**
- **State**: Tracks main conversation flow, rolling memory, temporary clarification state, sub-questions, and answers; clarification replies are discarded after the final answer
- **AgentState**: Manages individual execution, retrieval keys, actual tool contexts, and iteration tracking
- **QueryAnalysis**: Structured output for query rewriting and clarity checking

**State management:**
- `accumulate_or_reset`: Custom reducer for agent answers (allows reset)
- `set_union`: Custom reducer for retrieval keys (merges sets via union)
- `append_unique`: Preserves actual retrieval outputs across context-compression cycles
- Inherits from `MessagesState` for conversation history

**Documentation:**
- [LangGraph State](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)

In [ ]:
from langgraph.graph import MessagesState
from pydantic import BaseModel, Field
from typing import List, Annotated, Set
import operator

def accumulate_or_reset(existing: List[dict], new: List[dict]) -> List[dict]:
    if new and any(item.get('__reset__') for item in new):
        return []
    return existing + new

def set_union(a: Set[str], b: Set[str]) -> Set[str]:
    return a | b

def append_unique(existing: List[str], new: List[str]) -> List[str]:
    return list(dict.fromkeys(existing + new))

class State(MessagesState):
    """State for main agent graph"""
    questionIsClear: bool = False
    conversation_summary: str = ""
    originalQuery: str = ""
    pendingQuery: str = ""
    pendingClarifications: List[str] = []
    rewrittenQuestions: List[str] = []
    agent_answers: Annotated[List[dict], accumulate_or_reset] = []

class AgentState(MessagesState):
    """State for individual agent subgraph"""
    tool_call_count: Annotated[int, operator.add] = 0
    iteration_count: Annotated[int, operator.add] = 0
    question: str = ""
    question_index: int = 0
    context_summary: str = ""
    retrieval_keys: Annotated[Set[str], set_union] = set()
    retrieved_contexts: Annotated[List[str], append_unique] = []
    final_answer: str = ""
    agent_answers: List[dict] = []

class QueryAnalysis(BaseModel):
    is_clear: bool = Field(
        description="Indicates if the user's question is clear and answerable."
    )
    questions: List[str] = Field(
        description="List of rewritten, self-contained questions."
    )
    clarification_needed: str = Field(
        description="Explanation if the question is unclear."
    )

## 10. Agent Limits and Token Utilities

Define execution limits and token counting utilities for controlling agent loop behavior.

**What it does:**
- Sets hard limits on agent execution (tool calls, iterations)
- Estimates token usage from message lists to drive context management decisions

**Constants:**
- `MAX_TOOL_CALLS`: Maximum number of tool calls per agent run
- `MAX_ITERATIONS`: Maximum number of agent loop iterations
- `BASE_TOKEN_THRESHOLD`: Initial token threshold for context summarization
- `TOKEN_GROWTH_FACTOR`: Multiplier applied to threshold after each summarization

**Functions:**
- `estimate_context_tokens`: Uses cached `tiktoken` data when available and falls back to a deterministic character estimate when tokenizer data cannot be loaded offline

**Documentation:**
- [tiktoken](https://github.com/openai/tiktoken)

In [ ]:
import tiktoken
from functools import lru_cache

MAX_TOOL_CALLS = 8
MAX_ITERATIONS = 10
BASE_TOKEN_THRESHOLD = 2000
TOKEN_GROWTH_FACTOR = 0.9

@lru_cache(maxsize=1)
def _get_token_encoding():
    try:
        return tiktoken.encoding_for_model("gpt-4")
    except Exception:
        try:
            return tiktoken.get_encoding("cl100k_base")
        except Exception:
            return None

def estimate_context_tokens(messages: list) -> int:
    contents = [str(msg.content) for msg in messages if hasattr(msg, "content") and msg.content]
    encoding = _get_token_encoding()
    if encoding is None:
        return sum(max(1, len(content) // 4) for content in contents)
    return sum(len(encoding.encode(content)) for content in contents)

## 11. Graph Nodes and Logic

Implement node functions that define the behavior of the agentic workflow.

**What it does:**

### Core Nodes:
1. **summarize_history**: Maintains rolling memory by summarizing older conversation, deleting summarized/tool messages, and keeping bounded raw chat history
2. **rewrite_query**: Rewrites and optionally splits the user query into sub-questions while using the rolling summary and recent exchanges for context
3. **request_clarification**: Interrupt point for human-in-the-loop when the query is ambiguous
4. **orchestrator**: Main agent loop — invokes the LLM with tools, injects compressed context when available, and forces an initial retrieval call
5. **fallback_response**: Generates a best-effort answer directly from all retrieved tool data when the agent loop exhausts its budget
6. **compress_context**: Summarizes and compresses the current message history and prior context summary into a single block, appending a log of already-executed retrievals to prevent redundant calls; removes compressed messages from state
7. **collect_answer**: Extracts the final assistant answer from the agent message history and packages it with its index for aggregation
8. **aggregate_answers**: Sorts and merges all sub-question answers, then synthesizes them into a single coherent response matching the original user query

### Routing Logic:
- **route_after_rewrite**: Routes to `request_clarification` if the query is unclear; otherwise uses `Send` to spawn parallel `agent` subgraphs — one per rewritten sub-question
- **should_compress_context**: Uses `Command` to conditionally route to `compress_context` (when token usage exceeds a dynamic threshold) or back to `orchestrator`; also tracks all retrieval keys across iterations to avoid repeated tool calls

**Documentation:**
- [LangGraph Nodes](https://docs.langchain.com/oss/python/langgraph/graph-api#nodes)
- [LangGraph Edges](https://docs.langchain.com/oss/python/langgraph/graph-api#edges)
- [LangGraph Send API](https://docs.langchain.com/oss/python/langgraph/graph-api#send)
- [LangGraph Command API](https://docs.langchain.com/oss/python/langgraph/graph-api#command)

In [ ]:
from langgraph.types import Send, Command
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, RemoveMessage, ToolMessage
from typing import Literal, Set

MAIN_HISTORY_MESSAGES_TO_KEEP = 4
if MAIN_HISTORY_MESSAGES_TO_KEEP < 2:
    raise ValueError("MAIN_HISTORY_MESSAGES_TO_KEEP must be at least 2.")
PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP = max(MAIN_HISTORY_MESSAGES_TO_KEEP - 1, 0)

def _is_plain_conversation_message(msg) -> bool:
    return (
        isinstance(msg, (HumanMessage, AIMessage))
        and not getattr(msg, "tool_calls", None)
        and not getattr(msg, "name", None)
    )

def _name_internal_message(message, name):
    """Tag a subgraph-only message so it is not treated as chat history."""
    return message.model_copy(update={"name": name})

def _retrieval_contexts(messages) -> list[str]:
    contexts = []
    ignored_prefixes = (
        "NO_RELEVANT_CHUNKS",
        "NO_PARENT_DOCUMENT",
        "RETRIEVAL_ERROR:",
        "PARENT_RETRIEVAL_ERROR:",
    )
    for message in messages:
        if not isinstance(message, ToolMessage):
            continue
        content = str(message.content).strip()
        if content and not content.startswith(ignored_prefixes):
            parts = content.split(CHILD_CHUNK_SEPARATOR) if message.name == "search_child_chunks" else [content]
            contexts.extend(part for part in parts if part)
    return list(dict.fromkeys(contexts))

def _format_conversation(messages) -> str:
    lines = []
    for msg in messages:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)

def _remove_messages_not_in(messages, keep_ids):
    removals = []
    for msg in messages:
        msg_id = getattr(msg, "id", None)
        if isinstance(msg, SystemMessage) or not msg_id:
            continue
        if msg_id not in keep_ids:
            removals.append(RemoveMessage(id=msg_id))
    return removals

def _recent_conversation(messages, pending_query="") -> list:
    """Return recent context before the current user message."""
    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    recent_messages = plain_messages[:-1]

    if pending_query:
        for index in range(len(recent_messages) - 1, -1, -1):
            msg = recent_messages[index]
            if isinstance(msg, HumanMessage) and str(msg.content).strip() == pending_query:
                return recent_messages[:index]

    return recent_messages

def summarize_history(state: State):
    """Roll older chat into a summary and keep only bounded raw history."""
    messages = state.get("messages", [])
    updates = {"agent_answers": [{"__reset__": True}]}

    if not messages:
        return updates

    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    keep_count = PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP
    messages_to_summarize = plain_messages[:-keep_count] if len(plain_messages) > keep_count else []
    keep_ids = {getattr(msg, "id", None) for msg in plain_messages[-keep_count:]}
    keep_ids.discard(None)

    removals = _remove_messages_not_in(messages, keep_ids)
    if removals:
        updates["messages"] = removals

    if not messages_to_summarize:
        return updates

    existing_summary = state.get("conversation_summary", "").strip()
    conversation = "Existing summary:\n"
    conversation += f"{existing_summary or '(none)'}\n\n"
    conversation += "New messages to merge into the summary:\n"
    conversation += _format_conversation(messages_to_summarize)

    summary_response = llm.invoke([
        SystemMessage(content=get_conversation_summary_prompt()),
        HumanMessage(content=conversation),
    ])
    updates["conversation_summary"] = summary_response.content.strip()
    return updates

def rewrite_query(state: State):
    """Analyzes user query and rewrites it for clarity, optionally using conversation context."""
    last_message = state["messages"][-1]
    current_query = str(last_message.content).strip()
    conversation_summary = state.get("conversation_summary", "").strip()
    pending_query = state.get("pendingQuery", "").strip()
    pending_clarifications = state.get("pendingClarifications", [])
    recent_messages = _recent_conversation(state["messages"], pending_query)

    context_parts = []
    if conversation_summary:
        context_parts.append(f"Conversation Summary:\n{conversation_summary}")
    if recent_messages:
        context_parts.append(f"Recent Conversation:\n{_format_conversation(recent_messages)}")

    if pending_query:
        clarifications = [*pending_clarifications, current_query]
        clarification_text = "\n".join(
            f"{index}. {value}" for index, value in enumerate(clarifications, start=1)
        )
        context_parts.append(
            f"Unresolved User Query:\n{pending_query}\n\n"
            f"User Clarifications:\n{clarification_text}"
        )
        original_query = f"{pending_query}\nClarifications:\n{clarification_text}"
    else:
        clarifications = []
        context_parts.append(f"User Query:\n{current_query}")
        original_query = current_query

    context_section = "\n\n".join(context_parts)
    llm_with_structure = llm.with_structured_output(QueryAnalysis)
    response = llm_with_structure.invoke([SystemMessage(content=get_rewrite_query_prompt()), HumanMessage(content=context_section)])
    clarification_message_update = (
        [_name_internal_message(last_message, "clarification_response")]
        if pending_query else []
    )

    if response.questions and response.is_clear:
        return {
            "questionIsClear": True,
            "originalQuery": original_query,
            "pendingQuery": "",
            "pendingClarifications": [],
            "rewrittenQuestions": response.questions,
            "messages": clarification_message_update,
        }

    clarification = response.clarification_needed if response.clarification_needed and len(response.clarification_needed.strip()) > 10 else "I need more information to understand your question."
    return {
        "questionIsClear": False,
        "originalQuery": "",
        "pendingQuery": pending_query or current_query,
        "pendingClarifications": clarifications,
        "rewrittenQuestions": [],
        "messages": clarification_message_update + [
            AIMessage(content=clarification, name="clarification")
        ],
    }

def request_clarification(state: State):
    """Placeholder node for human-in-the-loop interruption."""
    return {}

def route_after_rewrite(state: State) -> Literal["request_clarification", "agent"]:
    if not state.get("questionIsClear", False):
        return "request_clarification"
    return [
        Send("agent", {"question": query, "question_index": idx, "messages": []})
        for idx, query in enumerate(state["rewrittenQuestions"])
    ]

def orchestrator(state: AgentState):
    """Main agent logic: decides when to search, retrieve, and answer."""
    context_summary = state.get("context_summary", "").strip()
    sys_msg = SystemMessage(content=get_orchestrator_prompt())
    summary_injection = (
        [HumanMessage(content=f"[COMPRESSED CONTEXT FROM PRIOR RESEARCH]\n\n{context_summary}")]
        if context_summary else []
    )

    if not state.get("messages"):
        human_msg = HumanMessage(content=state["question"], name="agent_question")
        force_search = HumanMessage(content="YOU MUST CALL 'search_child_chunks' AS THE FIRST STEP TO ANSWER THIS QUESTION.")
        response = llm_with_tools.invoke([sys_msg] + summary_injection + [human_msg, force_search])
        response = _name_internal_message(response, "agent_response")
        return {"messages": [human_msg, response], "tool_call_count": len(response.tool_calls or []), "iteration_count": 1}

    response = llm_with_tools.invoke([sys_msg] + summary_injection + state["messages"])
    response = _name_internal_message(response, "agent_response")
    tool_calls = response.tool_calls if hasattr(response, "tool_calls") else []
    return {"messages": [response], "tool_call_count": len(tool_calls) if tool_calls else 0, "iteration_count": 1}

def route_after_orchestrator_call(state: AgentState) -> Literal["tools", "fallback_response", "collect_answer"]:
    iteration = state.get("iteration_count", 0)
    tool_count = state.get("tool_call_count", 0)

    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", None) or []

    if not tool_calls:
        return "collect_answer"

    # Accept a final answer at the iteration boundary, but do not execute
    # tool calls that would exceed the configured research budget.
    if iteration >= MAX_ITERATIONS or tool_count > MAX_TOOL_CALLS:
        return "fallback_response"

    return "tools"

def fallback_response(state: AgentState):
    """Generate a best-effort answer using all retrieved information when limits are reached."""
    seen = set()
    unique_contents = []
    for m in state["messages"]:
        if isinstance(m, ToolMessage) and m.content not in seen:
            unique_contents.append(m.content)
            seen.add(m.content)

    context_summary = state.get("context_summary", "").strip()

    context_parts = []
    if context_summary:
        context_parts.append(f"## Compressed Research Context (from prior iterations)\n\n{context_summary}")
    if unique_contents:
        context_parts.append(
            "## Retrieved Data (current iteration)\n\n" +
            "\n\n".join(f"--- DATA SOURCE {i} ---\n{content}" for i, content in enumerate(unique_contents, 1))
        )

    context_text = "\n\n".join(context_parts) if context_parts else "No data was retrieved from the documents."
    prompt_content = (
        f"USER QUERY: {state.get('question')}\n\n"
        f"{context_text}\n\n"
        f"INSTRUCTION:\nProvide the best possible answer using only the data above."
    )
    response = llm.invoke([SystemMessage(content=get_fallback_response_prompt()), HumanMessage(content=prompt_content)])
    response = _name_internal_message(response, "agent_response")
    return {"messages": [response]}

def should_compress_context(state: AgentState) -> Command[Literal["compress_context", "orchestrator"]]:
    messages = state["messages"]

    new_ids: Set[str] = set()
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                if tc["name"] == "retrieve_parent_chunks":
                    raw = tc["args"].get("parent_id") or tc["args"].get("id") or tc["args"].get("ids") or []
                    if isinstance(raw, str):
                        new_ids.add(f"parent::{raw}")
                    else:
                        new_ids.update(f"parent::{r}" for r in raw)

                elif tc["name"] == "search_child_chunks":
                    query = tc["args"].get("query", "")
                    if query:
                        new_ids.add(f"search::{query}")
            break

    updated_ids = state.get("retrieval_keys", set()) | new_ids

    current_token_messages = estimate_context_tokens(messages)
    current_token_summary = estimate_context_tokens([HumanMessage(content=state.get("context_summary", ""))])
    current_tokens = current_token_messages + current_token_summary
    max_allowed = BASE_TOKEN_THRESHOLD + int(current_token_summary * TOKEN_GROWTH_FACTOR)

    goto = "compress_context" if current_tokens > max_allowed else "orchestrator"
    return Command(
        update={
            "retrieval_keys": updated_ids,
            "retrieved_contexts": _retrieval_contexts(messages),
        },
        goto=goto,
    )

def compress_context(state: AgentState):
    """Compress retrieval history into a concise summary for future iterations."""
    messages = state["messages"]
    existing_summary = state.get("context_summary", "").strip()

    if not messages:
        return {}

    conversation_text = f"USER QUESTION:\n{state.get('question')}\n\nConversation to compress:\n\n"
    if existing_summary:
        conversation_text += f"[PRIOR COMPRESSED CONTEXT]\n{existing_summary}\n\n"

    for msg in messages[1:]:
        if isinstance(msg, AIMessage):
            tool_calls_info = ""
            if getattr(msg, "tool_calls", None):
                calls = ", ".join(f"{tc['name']}({tc['args']})" for tc in msg.tool_calls)
                tool_calls_info = f" | Tool calls: {calls}"
            conversation_text += f"[ASSISTANT{tool_calls_info}]\n{msg.content or '(tool call only)'}\n\n"
        elif isinstance(msg, ToolMessage):
            tool_name = getattr(msg, "name", "tool")
            conversation_text += f"[TOOL RESULT — {tool_name}]\n{msg.content}\n\n"

    summary_response = llm.invoke([SystemMessage(content=get_context_compression_prompt()), HumanMessage(content=conversation_text)])
    new_summary = summary_response.content

    retrieved_ids: Set[str] = state.get("retrieval_keys", set())
    if retrieved_ids:
        parent_ids = sorted(r for r in retrieved_ids if r.startswith("parent::"))
        search_queries = sorted(r.replace("search::", "") for r in retrieved_ids if r.startswith("search::"))

        block = "\n\n---\n**Already executed (do NOT repeat):**\n"
        if parent_ids:
            block += "Parent chunks retrieved:\n" + "\n".join(f"- {p.replace('parent::', '')}" for p in parent_ids) + "\n"
        if search_queries:
            block += "Search queries already run:\n" + "\n".join(f"- {q}" for q in search_queries) + "\n"
        new_summary += block

    return {"context_summary": new_summary, "messages": [RemoveMessage(id=m.id) for m in messages[1:]]}

def collect_answer(state: AgentState):
    last_message = state["messages"][-1]
    is_valid = isinstance(last_message, AIMessage) and last_message.content and not last_message.tool_calls
    answer = last_message.content if is_valid else "Unable to generate an answer."
    return {
        "final_answer": answer,
        "agent_answers": [{
            "index": state["question_index"],
            "question": state["question"],
            "answer": answer,
            "contexts": state.get("retrieved_contexts", []),
        }]
    }

def aggregate_answers(state: State):
    messages = state.get("messages", [])
    plain_messages = [msg for msg in messages if _is_plain_conversation_message(msg)]
    keep_ids = {getattr(msg, "id", None) for msg in plain_messages[-PRE_ANSWER_HISTORY_MESSAGES_TO_KEEP:]}
    keep_ids.discard(None)
    removals = _remove_messages_not_in(messages, keep_ids)

    if not state.get("agent_answers"):
        return {"messages": removals + [AIMessage(content="No answers were generated.")]}

    sorted_answers = sorted(state["agent_answers"], key=lambda x: x["index"])

    formatted_answers = ""
    for i, ans in enumerate(sorted_answers, start=1):
        formatted_answers += (f"\nRetrieved response {i}:\n"f"{ans['answer']}\n")

    user_message = HumanMessage(content=f"""Original user question: {state["originalQuery"]}\nRetrieved answers:{formatted_answers}""")
    synthesis_response = llm.invoke([SystemMessage(content=get_aggregation_prompt()), user_message])
    return {"messages": removals + [AIMessage(content=synthesis_response.content)]}


## 12. LangGraph Construction

Construct the agentic workflow using LangGraph's state machine.

**What it does:**
- Builds a hierarchical graph with a main flow and a compiled agent subgraph
- **Agent subgraph**: Handles individual sub-question retrieval and reasoning in a tool loop, with context compression and fallback handling
- **Main graph**: Orchestrates conversation flow, query analysis, parallel agent execution, and response aggregation

**Agent Subgraph Flow:**
1. START → `orchestrator` (LLM + tools)
2. Route: tool call → `tools` → `should_compress_context` → compress or loop back to `orchestrator`
3. Route: iteration limit hit → `fallback_response` → `collect_answer`
4. Route: final answer ready → `collect_answer` → END

**Main Graph Flow:**
1. START → `summarize_history`
2. `rewrite_query` — rewrites and validates the query
3. Route: unclear → `request_clarification` (interrupt) → back to `rewrite_query` | clear → spawn parallel `agent` subgraphs via `Send`
4. All agents complete → `aggregate_answers`
5. END

**Human-in-the-loop:** Graph interrupts *before* `request_clarification` if the query is unclear, resuming at `rewrite_query` once the user provides input

**Key design decisions:**
- `should_compress_context` is a node (not just an edge) because it uses `Command` to conditionally update state and route in a single step
- `agent_subgraph` is compiled independently and embedded as a node in the main graph, enabling clean state isolation per sub-question
- `InMemorySaver` checkpointer enables short-term memory across conversation turns

**Documentation:**
- [LangGraph StateGraph](https://docs.langchain.com/oss/python/langgraph/graph-api#stategraph)
- [LangGraph Short-term Memory](https://docs.langchain.com/oss/python/langgraph/add-memory#add-short-term-memory)

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from IPython.display import Image, display

# Initialize checkpointer
checkpointer = InMemorySaver()

# Build agent subgraph (handles individual questions)
agent_builder = StateGraph(AgentState)
agent_builder.add_node(orchestrator)
agent_builder.add_node("tools", ToolNode([search_child_chunks, retrieve_parent_chunks]))
agent_builder.add_node(compress_context)
agent_builder.add_node(fallback_response)
agent_builder.add_node(should_compress_context)
agent_builder.add_node(collect_answer)

agent_builder.add_edge(START, "orchestrator")
agent_builder.add_conditional_edges("orchestrator", route_after_orchestrator_call, {"tools": "tools", "fallback_response": "fallback_response", "collect_answer": "collect_answer"})
agent_builder.add_edge("tools", "should_compress_context")
agent_builder.add_edge("compress_context", "orchestrator")
agent_builder.add_edge("fallback_response", "collect_answer")
agent_builder.add_edge("collect_answer", END)
agent_subgraph = agent_builder.compile()

# Build main graph (orchestrates workflow)
graph_builder = StateGraph(State)
graph_builder.add_node(summarize_history)
graph_builder.add_node(rewrite_query)
graph_builder.add_node(request_clarification)
graph_builder.add_node("agent", agent_subgraph)
graph_builder.add_node(aggregate_answers)

graph_builder.add_edge(START, "summarize_history")
graph_builder.add_edge("summarize_history", "rewrite_query")
graph_builder.add_conditional_edges("rewrite_query", route_after_rewrite)
graph_builder.add_edge("request_clarification", "rewrite_query")
graph_builder.add_edge(["agent"], "aggregate_answers")
graph_builder.add_edge("aggregate_answers", END)

# Compile graph
agent_graph = graph_builder.compile(checkpointer=checkpointer, interrupt_before=["request_clarification"])

display(Image(agent_graph.get_graph(xray=True).draw_mermaid_png()))
print("✓ Agent graph compiled successfully.")

## 12.5 Observability (Optional)

Track LLM calls, tool usage, and graph execution with [Langfuse](https://langfuse.com).

This step is optional. If you skip it, the chat interface works the same way, just without tracing.

Set these environment variables before running the next cell, or edit `project/.env.example` and copy it to `project/.env` when using the modular app:

```bash
LANGFUSE_ENABLED=true
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_BASE_URL=https://cloud.langfuse.com
```

For the conceptual tracing guide and platform overview, see the companion [observability notebook](observability.ipynb).

In [ ]:
import os
from langfuse import get_client
from langfuse.langchain import CallbackHandler

LANGFUSE_ENABLED = os.environ.get("LANGFUSE_ENABLED", "false").lower() == "true"
langfuse_handler = None
langfuse = None

if LANGFUSE_ENABLED:
    langfuse = get_client()
    try:
        if langfuse.auth_check():
            print("Langfuse client is authenticated and ready!")
            langfuse_handler = CallbackHandler()
        else:
            print("Authentication failed. Please check your credentials and host.")
    except Exception as e:
        print(f"Error during Langfuse authentication: {e}")
else:
    print("Langfuse tracing disabled. Set LANGFUSE_ENABLED=true to enable it.")

## 13. Gradio Interface

Create an interactive web interface for chatting with the RAG agent with real-time streaming.

**What it does:**
- Provides a streaming chat interface using Gradio
- Manages conversation threads with unique IDs
- Handles human-in-the-loop interactions seamlessly
- Automatically resumes interrupted workflows when the user provides clarification

**Streaming behavior by node type:**

| Node | Behavior |
|---|---|
| `rewrite_query` | Silenced — JSON parsed and rendered as readable markdown in a collapsible block |
| `summarize_history` | Displayed in a collapsible block with title "📋 Chat History Summary" |
| Tool calls | Shown as collapsible blocks with the tool name (e.g. `🛠️ search_child_chunks`) |
| Tool results | Injected into the corresponding tool block (truncated to 300 chars) |
| `aggregate_answers` | Streamed token by token as the final visible response |
| `orchestrator`, `compress_context`, `fallback_response` | Kept internal; their raw model output is not shown as the final answer |

**Key helpers:**
- `make_message()` — builds Gradio message dicts with optional collapsible metadata
- `find_msg_idx()` — looks up an existing message by node name to update it in place
- `parse_rewrite_json()` — extracts and parses the structured output from `rewrite_query`
- `format_rewrite_content()` — converts the JSON into human-readable markdown (✅ clear / ❓ unclear + rewritten queries)

**Features:**
- Token-by-token streaming via `stream_mode="messages"`
- Collapsible thought blocks for system nodes and tool calls
- Clarification message surfaced as a plain visible message when query is unclear
- Thread-based conversation persistence
- Clear session functionality with checkpointer cleanup
- Citrus theme for modern UI

**Note:** For a complete end-to-end pipeline with document ingestion UI, refer to the full application in the project repository.

**Documentation:**
- [Gradio ChatInterface](https://www.gradio.app/docs/gradio/chatinterface)
- [Gradio Chatbot](https://www.gradio.app/docs/gradio/chatbot)
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)

In [ ]:
import gradio as gr
import uuid
import json
import re
from langchain_core.messages import HumanMessage, AIMessageChunk, ToolMessage
import logging
# Suppress OTel "Failed to detach context" warning caused by generator/context interaction.
# Tracing is unaffected.
# Known bug: https://github.com/open-telemetry/opentelemetry-python/issues/2606
class _SuppressOtelDetachWarning(logging.Filter):
    def filter(self, record):
        return "Failed to detach context" not in record.getMessage()

logging.getLogger("opentelemetry.context").addFilter(_SuppressOtelDetachWarning())

# ── Constants ────────────────────────────────────────────────────────────────

SYSTEM_NODES  = {"summarize_history", "rewrite_query"}
FINAL_RESPONSE_NODES = {"aggregate_answers"}

SYSTEM_NODE_CONFIG = {
    "rewrite_query":     {"title": "🔍 Query Analysis & Rewriting"},
    "summarize_history": {"title": "📋 Chat History Summary"},
}

# ── Helpers ───────────────────────────────────────────────────────────────────

def create_thread_id():
    return {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 50}

def clear_session():
    global config
    agent_graph.checkpointer.delete_thread(config["configurable"]["thread_id"])
    config = create_thread_id()
    if langfuse is not None:
        try:
            langfuse.flush()
        except Exception:
            pass

def make_message(content, *, title=None, node=None):
    """Build a Gradio chat message dict, with optional collapsible metadata."""
    msg = {"role": "assistant", "content": content}
    if title or node:
        msg["metadata"] = {k: v for k, v in {"title": title, "node": node}.items() if v}
    return msg

def find_msg_idx(messages, node):
    """Return index of first message whose metadata.node matches, or None."""
    return next(
        (i for i, m in enumerate(messages) if m.get("metadata", {}).get("node") == node),
        None
    )

def parse_rewrite_json(buffer):
    """Try to parse structured JSON from rewrite_query buffer. Returns dict or None."""
    match = re.search(r'\{.*\}', buffer, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except Exception:
        return None

def format_rewrite_content(buffer):
    """Convert rewrite_query JSON buffer into readable markdown."""
    data = parse_rewrite_json(buffer)
    if not data:
        return "⏳ Analyzing query..."

    if data.get("is_clear"):
        lines = ["✅ **Query is clear**"]
        if data.get("questions"):
            lines += ["\n**Rewritten queries:**"] + [f"- {q}" for q in data["questions"]]
    else:
        lines = ["❓ **Query is unclear**"]
        clarification = data.get("clarification_needed", "")
        if clarification and clarification.strip().lower() != "no":
            lines.append(f"\nClarification needed: *{clarification}*")

    return "\n".join(lines)

# ── Observability ───────────────────────────────────────────────────────────────
# If the observability cell was skipped, default to no tracing
if "langfuse_handler" not in dir():
    langfuse_handler = None
if "langfuse" not in dir():
    langfuse = None

# ── Streaming ─────────────────────────────────────────────────────────────────

def chat(message, history):
    current_state = agent_graph.get_state(config)

    if current_state.next:
        agent_graph.update_state(config, {"messages": [HumanMessage(content=message.strip())]})
        stream_input = None
    else:
        stream_input = {"messages": [HumanMessage(content=message.strip())]}

    # Add Langfuse handler if observability is enabled
    run_config = {**config, **({"callbacks": [langfuse_handler]} if langfuse_handler else {})}

    response_messages = []
    active_tool_calls  = {}
    system_node_buffer = {}

    for chunk, metadata in agent_graph.stream(stream_input, config=run_config, stream_mode="messages"):
        node = metadata.get("langgraph_node", "")

        # ── System nodes (summarize_history, rewrite_query) ──────────────────
        if node in SYSTEM_NODES and isinstance(chunk, AIMessageChunk) and chunk.content:
            system_node_buffer[node] = system_node_buffer.get(node, "") + chunk.content
            buffer = system_node_buffer[node]
            idx    = find_msg_idx(response_messages, node)
            title  = SYSTEM_NODE_CONFIG[node]["title"]

            content = format_rewrite_content(buffer) if node == "rewrite_query" else buffer

            if idx is None:
                response_messages.append(make_message(content, title=title, node=node))
            else:
                response_messages[idx]["content"] = content

            # rewrite_query: surface clarification as a plain (non-collapsible) message
            if node == "rewrite_query":
                data = parse_rewrite_json(buffer) or {}
                clarification = data.get("clarification_needed", "")
                if not data.get("is_clear") and clarification.strip().lower() not in ("", "no"):
                    cidx = find_msg_idx(response_messages, "clarification")
                    if cidx is None:
                        response_messages.append(make_message(clarification, node="clarification"))
                    else:
                        response_messages[cidx]["content"] = clarification

            yield response_messages
            continue

        # ── Tool calls ────────────────────────────────────────────────────────
        if hasattr(chunk, "tool_calls") and chunk.tool_calls:
            for tc in chunk.tool_calls:
                if tc.get("id") and tc["id"] not in active_tool_calls:
                    response_messages.append(make_message(f"Running `{tc['name']}`...", title=f"🛠️ {tc['name']}"))
                    active_tool_calls[tc["id"]] = len(response_messages) - 1

        # ── Tool results ──────────────────────────────────────────────────────
        elif isinstance(chunk, ToolMessage):
            if (idx := active_tool_calls.get(chunk.tool_call_id)) is not None:
                preview = str(chunk.content)[:300]
                suffix  = "\n..." if len(str(chunk.content)) > 300 else ""
                response_messages[idx]["content"] = f"```\n{preview}{suffix}\n```"

        # ── LLM tokens (final response) ───────────────────────────────────────
        elif isinstance(chunk, AIMessageChunk) and chunk.content and node in FINAL_RESPONSE_NODES:
            last = response_messages[-1] if response_messages else None
            if not (last and last.get("role") == "assistant" and "metadata" not in last):
                response_messages.append(make_message(""))
            response_messages[-1]["content"] += chunk.content

        else:
            continue

        yield response_messages

# ── App ───────────────────────────────────────────────────────────────────────

config = create_thread_id()

with gr.Blocks() as demo:
    chatbot = gr.Chatbot(
        height=720,
        placeholder="<strong>Ask me anything!</strong><br><em>I'll search, reason, and act to give you the best answer :)</em>",
        show_label=False
    )
    chatbot.clear(clear_session)
    gr.ChatInterface(fn=chat, chatbot=chatbot)

print("\nLaunching application...")
demo.launch(theme=gr.themes.Citrus())